In [ ]:
import subprocess, sys, os, glob, shutil, time, json
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True); print(r.stdout[-1500:], r.stderr[-800:], flush=True); return r
W = "/kaggle/working/w"; shutil.rmtree(W, ignore_errors=True); os.makedirs(W)
src = [p for p in glob.glob("/kaggle/input/**/luoi-cpu-3-11", recursive=True) if os.path.isdir(p)]; print(src); S = src[0]
for z in glob.glob(S + "/*.zip"): sh(f"cd {W} && unzip -o -q {z} -d {W}/{os.path.basename(z)[:-4]}")
for d in ("harness", "runs"):
    if os.path.isdir(f"{S}/{d}"): shutil.copytree(f"{S}/{d}", f"{W}/{d}", dirs_exist_ok=True)
sh(f"find {W} -type f | wc -l; ls {W} {W}/harness; wc -l {W}/runs/luoi/test_2_11/*.jsonl")
sh(f"{sys.executable} -m pip install -q scipy spacy 2>&1 | tail -2; {sys.executable} -m spacy download en_core_web_sm 2>&1 | tail -1")


In [ ]:
KQ = {}
def chay(ten, lenh):
    t0 = time.time(); log = f"{W}/{ten}.log"
    p = subprocess.Popen(f"cd {W}/harness && {sys.executable} -u {lenh} > {log} 2>&1", shell=True)
    while p.poll() is None:
        time.sleep(60); print(time.strftime("%H:%M"), ten, open(log, errors="ignore").read()[-200:].replace("\n", " | "), flush=True)
    KQ[ten] = p.returncode; print("=====", ten, "ma thoat", p.returncode, f"{time.time()-t0:.0f} s", flush=True); print(open(log, errors="ignore").read()[-5000:], flush=True)
chay("phan_ra", "phan_ra_luoi.py")
chay("doc_luoi", "doc_luoi.py")
chay("log_luoi", "log_luoi.py")
chay("hinh_thuc", "hinh_thuc_cau.py --out ../runs/luoi/test_2_11/hinh_thuc.json --them SCST+SPICE=runs/luoi/test_2_11/score_scst_spice_101_test_raw.jsonl --them GRPO+CIDEr-D=runs/luoi/test_2_11/score_grpo_cider_101_test_raw.jsonl --them GRPO+METEOR=runs/luoi/test_2_11/score_grpo_meteor_101_test_raw.jsonl --them GRPO+ROUGE-L=runs/luoi/test_2_11/score_grpo_rougel_101_test_raw.jsonl")
print(KQ, flush=True)


In [ ]:
for f in glob.glob(f"{W}/runs/luoi/test_2_11/*.json") + glob.glob(f"{W}/*.log"):
    shutil.copyfile(f, "/kaggle/working/" + os.path.basename(f))
shutil.rmtree(W); print(os.listdir("/kaggle/working"), KQ, flush=True)
assert all(v == 0 for v in KQ.values()), KQ
